# v5 无旧上限五会话锁定分支机械试验

冻结 `plan.json` 后只选 2019-12-19 09:48 完成分钟。父本已于09:47成交；HOLD与+0.10 SOXX事件在09:50后按相同五会话锁定规则运行。核验Qlib分钟价、整股Tiered、独立现金/高水位和终端净值，不据单事件收益选择因子。

In [1]:
import os,json,hashlib,warnings
from pathlib import Path
os.environ["MLFLOW_ALLOW_FILE_STORE"]="true"
os.environ["MLFLOW_DISABLE_AGENT_HINT"]="1"
warnings.filterwarnings("ignore",message="The 'generic' unit for NumPy timedelta")
import numpy as np
import pandas as pd
import qlib
from qlib.data import D
from src.core.recorder_sources import load_verified_recorder_objects
from src.backtest.stage1_event_checkpoint import capture_after_open
from src.backtest.stage1_capfree_lock_pilot import replay_capfree_lock_pilot
from src.backtest.ibkr_tiered import settle_tiered_event
from src.workflow.stage1_dated_controller import DatedModel,DatedStage1Controllers
from src.workflow.stage1_f696_training_reference import prepare as prepare_f696_source
from src.workflow.intraday_recovery_direct_allocation import IntradayRecoveryDirectAllocationSpec
from src.workflow.stage1_open_action_value import Stage1OpenActionValueSpec
from copy import deepcopy
root=Path.cwd();card=root/"research/stage1_v5_capfree_lock_pilot_20261030";plan=json.loads((card/"plan.json").read_text())
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
parent=root/plan["parent_card"];old=root/plan["old_source_card"]
assert sha(parent/"summary.json")==plan["parent_summary_sha256"] and sha(parent/"verification_manifest.json")==plan["parent_manifest_sha256"]
assert sha(old/"plan.json")==plan["old_source_plan_sha256"]
assert sha(root/plan["checkpoint_code"])==plan["checkpoint_code_sha256"]
assert sha(root/plan["minute_provider_uri"]/"qlib_store_manifest.json")==plan["minute_provider_manifest_sha256"]
parent_plan=json.loads((parent/"plan.json").read_text());old_plan=json.loads((old/"plan.json").read_text())
prepared=prepare_f696_source(root=root,card=old,plan=old_plan)
qlib.init(provider_uri=str(root/plan["minute_provider_uri"]),region="us",kernels=1)
names=["analysis/native/dated_f696_0","analysis/controller_audit",*[f"models/open_{y}" for y in parent_plan["signal_years"]],*[f"models/recovery_{y}" for y in (2020,2021,2022)]]
objects,receipt=load_verified_recorder_objects(parent,expected_recorder_id=plan["parent_recorder_id"],artifacts=names,expected_manifest_sha256=plan["parent_manifest_sha256"])
book=objects["analysis/native/dated_f696_0"];frame=prepared["frame"]
assert len(frame)==1056 and book["audit"]["status"]=="passed"
open_models={y:DatedModel(objects[f"models/open_{y}"],f"{y-1}-12-31",f"{parent_plan['action_recorder_id']}:open_action_value_models/{y}") for y in parent_plan["signal_years"]}
recovery_models={}
for y in parent_plan["signal_years"]:
 if y in parent_plan["explicit_no_recovery_model_years"]:recovery_models[y]=DatedModel(None,f"{y-1}-12-31",f"explicit_no_model_{y}_insufficient_prior_teacher",explicit_no_model=True)
 else:recovery_models[y]=DatedModel(objects[f"models/recovery_{y}"],f"{y-1}-12-31",f"{plan['parent_recorder_id']}:models/recovery_{y}")
controller=DatedStage1Controllers(calendar=frame[["signal_date","trade_date"]],open_models=open_models,recovery_models=recovery_models)
pd.testing.assert_frame_equal(controller.audit(),objects["analysis/controller_audit"],check_dtype=False)
recovery_spec=IntradayRecoveryDirectAllocationSpec(root=root,**prepared["action_plan"]["recovery_spec"])
action_spec=Stage1OpenActionValueSpec(root=root,**prepared["action_plan"]["action_spec"])
print('dated parent and minute source verified',len(receipt))

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[22472:MainThread](2026-10-30 13:31:39,667) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[22472:MainThread](2026-10-30 13:31:39,668) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[22472:MainThread](2026-10-30 13:31:39,668) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/full_path_tiered_allocation_20261013')}


[22472:MainThread](2026-10-30 13:31:39,740) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[22472:MainThread](2026-10-30 13:31:39,741) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[22472:MainThread](2026-10-30 13:31:39,741) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/stage1_event_guard_fullminute_20261028')}


dated parent and minute source verified 13


In [2]:
day=pd.Timestamp(plan["event_day"]);bar_end=pd.Timestamp(plan["bar_end_at"]);decision=pd.Timestamp(plan["decision_at"]);fill=pd.Timestamp(plan["fill_at"])
checkpoint=capture_after_open(frame,event_day=day,controller_by_date=controller,recovery_spec=recovery_spec,action_spec=action_spec,selection=prepared["selection"],plan=parent_plan)
assert checkpoint.index>=0 and not book["independent_replay"].iloc[checkpoint.index].recovery_action
assert recovery_models[2019].explicit_no_model
last=frame.iloc[checkpoint.index+plan["horizon_sessions"]-1]
raw=D.features(["SOXL","SOXX"],["$open","$close","$raw_open","$price_valid"],start_time=str(day.date())+" 09:31:00",end_time=str(pd.Timestamp(last.trade_date).date())+" 16:00:00",freq="1min")
parts=[]
for symbol in ("SOXL","SOXX"):
 x=raw.xs(symbol,level="instrument").copy();x.columns=[symbol+"_"+v.removeprefix("$") for v in x.columns];parts.append(x)
quotes=pd.concat(parts,axis=1).sort_index()
assert quotes[["SOXL_price_valid","SOXX_price_valid"]].eq(1).all().all()
mark=quotes.loc[bar_end];mark_price=np.array([mark.SOXL_close,mark.SOXX_close],float)
nav=float(checkpoint.state.q@mark_price+checkpoint.state.cash)
w=np.r_[checkpoint.state.q*mark_price,checkpoint.state.cash]/nav
assert w[2]>plan["nominal_beta_increment"]
target=w+np.array([0.,plan["nominal_beta_increment"],-plan["nominal_beta_increment"]])
fillbar=quotes.loc[fill+pd.Timedelta(minutes=1)]
adjusted=np.array([fillbar.SOXL_open,fillbar.SOXX_open],float);raw_fill=np.array([fillbar.SOXL_raw_open,fillbar.SOXX_raw_open],float)
event={"trade_date":day,"bar_end_at":bar_end,"decision_at":decision,"fill_at":fill,"desired_weights":target,"adjusted_fill_prices":adjusted,"raw_fill_prices":raw_fill}
preflight=settle_tiered_event(checkpoint.state.q,checkpoint.state.cash,desired=target,adjusted_prices=adjusted,raw_prices=raw_fill,stamp=fill,fees=deepcopy(checkpoint.state.fees))
assert len(preflight["orders"])==1 and preflight["orders"][0]["symbol"]=="SOXX" and not preflight["orders"][0]["sell"]
result=replay_capfree_lock_pilot(frame,checkpoint,quotes,event=event,horizon_sessions=5)
hold=result["HOLD"];branch=result["EVENT"]
assert len(branch["orders"])==1 and not hold["orders"]
assert abs(float(hold["path"].iloc[0].next_open_nav)-float(book["independent_replay"].iloc[checkpoint.index].native_expected_nav))<plan["maximum_nav_error_usd"]
assert all(hold["path"].event_fee_usd.eq(0)) and branch["path"].event_fee_usd.iloc[1:].eq(0).all()
assert np.allclose(branch["ending_quantities"],preflight["quantities"],atol=0,rtol=0) and branch["ending_cash"]==preflight["cash"]
terminal_price=last[["soxl_exit_open","soxx_exit_open"]].to_numpy(float)
terminal_delta=float(branch["path"].iloc[-1].next_open_nav-hold["path"].iloc[-1].next_open_nav)
mechanical_delta=float((branch["ending_quantities"]-hold["ending_quantities"])@terminal_price+branch["ending_cash"]-hold["ending_cash"])
assert abs(terminal_delta-mechanical_delta)<plan["maximum_nav_error_usd"]
late=dict(event,bar_end_at=day+pd.Timedelta(hours=10,minutes=31),decision_at=day+pd.Timedelta(hours=10,minutes=32),fill_at=day+pd.Timedelta(hours=10,minutes=33))
try:replay_capfree_lock_pilot(frame,checkpoint,quotes,event=late,horizon_sessions=5)
except ValueError as exc:assert 'pre-recovery' in str(exc)
else:raise AssertionError('late event was accepted by early-only pilot')
old_cap=float(book["independent_replay"].iloc[checkpoint.index].current_beta_cap)
target_beta=float(3*target[0]+target[1])
audit={"experiment":plan["experiment"],"plan_sha256":sha(card/"plan.json"),"event_day":str(day.date()),"event_clock":str(bar_end)+"->"+str(decision)+"->"+str(fill),"horizon_sessions":5,"source_minute_rows":len(raw),"same_start_decision_nav":result["decision_nav"],"hold_day1_parent_next_open_error_usd":float(abs(hold["path"].iloc[0].next_open_nav-book["independent_replay"].iloc[checkpoint.index].native_expected_nav)),"event_orders":len(branch["orders"]),"old_parent_beta_cap_at_open":old_cap,"event_target_nominal_beta":target_beta,"event_target_exceeds_old_cap":bool(target_beta>old_cap+1e-10),"pilot_code_sha256":sha(root/"src/backtest/stage1_capfree_lock_pilot.py"),"event_raw_shares":int(branch["orders"][0]["raw_shares"]),"event_fee_usd":float(branch["orders"][0]["total"]),"event_cash_nonnegative":bool(branch["ending_cash"]>=0),"terminal_nav_closure_error_usd":abs(terminal_delta-mechanical_delta),"hold_ending_cash":hold["ending_cash"],"event_ending_cash":branch["ending_cash"],"hold_minute_close_max_drawdown":float(hold["path"].minute_close_max_drawdown.max()),"event_minute_close_max_drawdown":float(branch["path"].minute_close_max_drawdown.max()),"late_clock_rejected":True,"old_cap_used_in_lock":False,"new_model_fits":0,"new_native_candidate_accounts":0,"historical_research_passed":False,"decision":"single_early_event_capfree_five_session_mechanical_pass_not_bulk_labels"}
(card/"audit.json").write_text(json.dumps(audit,ensure_ascii=False,indent=2)+"\n")
print(json.dumps(audit,ensure_ascii=False,indent=2))

{
  "experiment": "stage1_v5_capfree_lock_pilot_20261030",
  "plan_sha256": "7eeddc0a710cf545ba07866da7f6282eeae87c19664d05ffdf7306a083e4bffb",
  "event_day": "2019-12-19",
  "event_clock": "2019-12-19 09:48:00->2019-12-19 09:49:00->2019-12-19 09:50:00",
  "horizon_sessions": 5,
  "source_minute_rows": 3540,
  "same_start_decision_nav": 159019.0495717724,
  "hold_day1_parent_next_open_error_usd": 0.0,
  "event_orders": 1,
  "old_parent_beta_cap_at_open": 1.3949364778479563,
  "event_target_nominal_beta": 1.4915449671921246,
  "event_target_exceeds_old_cap": true,
  "pilot_code_sha256": "f0f5d5d82baf9a12acfb9311589c54752e6a3d2c476454a2fa6383ab361741f2",
  "event_raw_shares": 64,
  "event_fee_usd": 0.55524925,
  "event_cash_nonnegative": true,
  "terminal_nav_closure_error_usd": 2.1827872842550278e-11,
  "hold_ending_cash": 31084.048158404443,
  "event_ending_cash": 15255.012440404444,
  "hold_minute_close_max_drawdown": 0.009570343031858108,
  "event_minute_close_max_drawdown": 0.010218

## 中文结论

**SOXX硬门：未运行任何新候选账户或0/2/5bp检验。** 2019-12-19事前固定09:48完成分钟→09:49决策→09:50成交，五会话的HOLD和+0.10 SOXX两支从同一逐层点时父本开盘状态与Tiered月度计数复制。增配支形成64股SOXX一笔整股买入，费用0.55524925美元，现金非负；目标名义beta1.491545高于旧父本上限1.394936，锁定分支不使用该上限。HOLD首日次开盘NAV与父本误差0，五会话终端自身股数/现金NAV闭合2.18e−11美元；晚于恢复时刻的事件被此早盘pilot明确拒绝。

两支均在成交后停止旧b472动作五会话，各自持有股数/现金、月度费用计数和按Qlib分钟收盘递推的高水位。单次分钟收盘最大回撤HOLD约0.957%、EVENT约1.022%，仅是机械样例，不能当信号或SOXX超额。2019明确无恢复模型；此模块尚不能覆盖10:32后、原恢复年、半日市、多动作、任意连续三资产目标，也没有完整候选账户。2021/2022结果未用于。开发调整价单独分红现金仍unknown、真实历史首收/成交仍needs_data。下一步扩展同域回放到所有合法分钟和连续目标，按逐日/逐年和费用核验后才冻结批量5会话标签及信息门。